# **Implementando modelos personalizados en Python: KNN, Regresión Lineal y Ofuscador de Datos** 🧮➡️🤖

## Objetivos académicos
- Comprender el papel del **científico de datos** en la construcción de modelos de ML desde cero, sin depender exclusivamente de librerías externas.
- Implementar de manera práctica **KNN** y **Regresión Lineal** utilizando clases en Python.
- Explorar cómo diseñar un **Ofuscador de datos** mediante transformaciones matriciales reversibles para proteger la información.


## Introducción
El científico de datos no solo aplica modelos ya disponibles en librerías, también debe entender su **fundamento matemático y computacional**.  
Esto le permite:
- Personalizar algoritmos.  
- Adaptarlos a necesidades específicas.  
- Validar su correcto funcionamiento.  

En esta sesión aprenderemos a **implementar modelos básicos desde cero** en Python, reforzando tanto el entendimiento matemático como la habilidad de programarlos en forma de clases.

In [28]:
import pandas as pd
import numpy as np
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, TransformerMixin

## Implementación de KNN (K-Nearest Neighbors) 👥📏
El algoritmo KNN clasifica un nuevo punto **comparando su distancia** con los puntos del conjunto de entrenamiento y tomando la **mayoría de los k vecinos más cercanos**.  
Su sencillez lo hace útil para comprender cómo las **distancias** definen fronteras de decisión en el espacio de características.

### Intuición del problema matricial

In [29]:
Data=np.array([[1,2],[2,3],[3,4],[6,6],[5,6]])
labels = np.array(["a","a","a","b","b"])
Cases=np.array([[3,3],[4,4]])
k=3
Data, Data.shape, Cases, Cases.shape

(array([[1, 2],
        [2, 3],
        [3, 4],
        [6, 6],
        [5, 6]]),
 (5, 2),
 array([[3, 3],
        [4, 4]]),
 (2, 2))

In [30]:
d=Cases[:, None, :] - Data[None, :, :]
d, d.shape

(array([[[ 2,  1],
         [ 1,  0],
         [ 0, -1],
         [-3, -3],
         [-2, -3]],
 
        [[ 3,  2],
         [ 2,  1],
         [ 1,  0],
         [-2, -2],
         [-1, -2]]]),
 (2, 5, 2))

In [31]:
distances=np.linalg.norm(d, axis=2)
distances

array([[2.23606798, 1.        , 1.        , 4.24264069, 3.60555128],
       [3.60555128, 2.23606798, 1.        , 2.82842712, 2.23606798]])

In [32]:
idx_k=np.argpartition(distances, kth=k-1, axis=1)[:, :k]
idx_k

array([[1, 2, 0],
       [2, 1, 4]])

In [33]:
labels[idx_k]

array([['a', 'a', 'a'],
       ['a', 'a', 'b']], dtype='<U1')

In [ ]:

class KNNClassifier(BaseEstimator, ClassifierMixin):

    """
    K-Nearest Neighbors.
    Parámetros
    ----------
    k : int
        Número de vecinos a considerar (k >= 1).
    metric : {"euclidean", "manhattan"}
        Métrica de distancia a utilizar.
    """

    def __init__(self, k=3, metric="euclidean"):
        # Validaciones básicas del init
        if not isinstance(k, int) or k < 1:
            raise ValueError("k debe ser un entero positivo (k >= 1).")
        metric = str(metric).lower()
        if metric not in {"euclidean", "manhattan"}:
            raise ValueError('metric debe ser "euclidean" o "manhattan".')
        self.k = k
        self.metric = metric
        # atributos que se setearán en fit
        self._X = None  # matriz de entrenamiento (n_samples, n_features)
        self._y = None  # etiquetas de entrenamiento (n_samples,)
        self.classes_ = None  # clases únicas en el orden interno
        self.inv_map = None # Etiquetas para el clasificador (0,1,2,...)

    def fit(self, X, y):
        """
        Guarda los datos de entrenamiento.
        - X: np.ndarray de forma (n_samples, n_features)
        - y: np.ndarray de forma (n_samples,)
        - Puede incluir validaciones de tipos/dimensiones.
        - Debe devolver self para cumplir el API de sklearn.
        """
        X, y = self._validate_inputs(X, y)
        if self.k > X.shape[0]:
            raise ValueError(f"k={self.k} no puede ser mayor que el número de muestras ({X.shape[0]}).")
        self._X = X
        self._y = y
        self.classes_ = np.unique(y)
        self.inv_map = {c: j for j, c in enumerate(self.classes_)}
        return self

    def predict(self, X):
        """
        Predice la clase para cada muestra en X.
        - Calcula distancias entre X y _X.
        - Identifica índices de los k vecinos más cercanos con np.argsort/argpartition.
        - Hace votación mayoritaria (resolver empates si ocurren).
        - Devuelve np.ndarray (n_samples_pred,) con etiquetas predichas.
        """
        self._check_is_fitted()
        X = self._validate_inputs(X, y=None)

        # Matriz de distancias (n_pred, n_train)
        D = self._pairwise_distances(X, self._X)

        # Índices de los k vecinos más cercanos por fila (argpartition para eficiencia)
        k = self.k

        # top-k (no ordenados)
        neigh_idx_part = np.argpartition(D, kth=k-1, axis=1)[:, :k]

        # Votación mayoritaria
        y_pred = np.empty(X.shape[0], dtype=self._y.dtype)
        for i in range(X.shape[0]):
            idx_k = neigh_idx_part[i]
            neigh_labels = self._y[idx_k]
            neigh_idx_labels = np.fromiter((self.inv_map[c] for c in neigh_labels), dtype=int, count=len(neigh_labels))
            counts = np.bincount(neigh_idx_labels, minlength=self.classes_.size)
            winner_local = int(np.argmax(counts))
            y_pred[i] = self.classes_[winner_local]
        return y_pred


    def _pairwise_distances(self, A, B):
        """
        Calcula distancias pairwise entre A (m, d) y B (n, d) según self.metric.
        Devuelve una matriz (m, n) con las distancias.
        """
        if self.metric == "euclidean":
            return self._euclidean(A, B)
        elif self.metric == "manhattan":
            return self._manhattan(A, B)
        else:
            # Esta rama no debería alcanzarse por la validación en __init__
            raise ValueError('metric debe ser "euclidean" o "manhattan".')

    def _euclidean(self, A, B):
        """
        Distancia Euclídea.
        Implementación vectorizada:
        - Para cada fila a en A y b en B, d(a,b) = ||a - b||_2
        - Usar broadcasting: np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
        """
        diff = A[:, None, :] - B[None, :, :]
        return np.linalg.norm(diff, axis=2)

    def _manhattan(self, A, B):
        """
        Distancia Manhattan L1 pairwise.
        Implementación vectorizada:
        - d(a,b) = sum(|a_i - b_i|)
        """
        return np.abs(A[:, None, :] - B[None, :, :]).sum(axis=2)

    # ----------------------
    # Métodos auxiliares (opcionales)
    # ----------------------

    def _check_is_fitted(self):
        """
        Verifica que fit() haya sido llamado (conforme a sklearn.utils.validation).
        """
        if self._X is None or self._y is None or self.classes_ is None:
            raise ValueError("Este KNNClassifier no está ajustado aún. Llama primero a fit(X, y).")

    def _validate_inputs(self, X, y=None):
        """
        Normaliza tipos y valida dimensiones/NaNs.
        - Si y no es None: validar longitud y tipos (clasificación).
        """
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X debe ser un arreglo 2D de forma (n_samples, n_features).")
        if y is None:
            return X
        y = np.asarray(y)
        if y.ndim != 1:
            raise ValueError("y debe ser un arreglo 1D de longitud n_samples.")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"Incompatibilidad de muestras: X tiene {X.shape[0]} filas y y tiene {y.shape[0]} elementos.")
        # (opcional) Chequeo de NaNs
        if np.isnan(X).any():
            raise ValueError("X contiene NaNs; limpia o imputa antes de usar KNN.")
        if np.isnan(y.astype(float, copy=False), where=False).any() if np.issubdtype(y.dtype, np.floating) else False:
            # Solo intentamos NaN-check si y es flotante; si es object/str, omitimos NaN check estricto
            raise ValueError("y contiene NaNs; limpia antes de usar KNN.")

        return X, y


In [35]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier as SkKNN

In [36]:
X, y = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    n_classes=3,
    class_sep=1.2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)


In [37]:
scaler = StandardScaler().fit(X_train)
X_train_sc = scaler.transform(X_train)
X_test_sc  = scaler.transform(X_test)


In [38]:
knn_eu = KNNClassifier(k=5, metric="euclidean").fit(X_train_sc, y_train)
y_pred_eu = knn_eu.predict(X_test_sc)
acc_eu = accuracy_score(y_test, y_pred_eu)

knn_l1 = KNNClassifier(k=5, metric="manhattan").fit(X_train_sc, y_train)
y_pred_l1 = knn_l1.predict(X_test_sc)
acc_l1 = accuracy_score(y_test, y_pred_l1)

print("=== KNN personalizado ===")
print(f"Accuracy (euclidean): {acc_eu:.4f}")
print(f"Accuracy (manhattan): {acc_l1:.4f}")
print("\nReporte (euclidean):")
print(classification_report(y_test, y_pred_eu, digits=4))
print("Matriz de confusión (euclidean):")
print(confusion_matrix(y_test, y_pred_eu))

=== KNN personalizado ===
Accuracy (euclidean): 0.8440
Accuracy (manhattan): 0.8360

Reporte (euclidean):
              precision    recall  f1-score   support

           0     0.8780    0.8571    0.8675        84
           1     0.8391    0.8690    0.8538        84
           2     0.8148    0.8049    0.8098        82

    accuracy                         0.8440       250
   macro avg     0.8440    0.8437    0.8437       250
weighted avg     0.8442    0.8440    0.8440       250

Matriz de confusión (euclidean):
[[72  4  8]
 [ 4 73  7]
 [ 6 10 66]]


In [39]:
sk_eu = SkKNN(n_neighbors=5, metric="minkowski", p=2).fit(X_train_sc, y_train)  # Euclidiana
y_pred_sk_eu = sk_eu.predict(X_test_sc)
acc_sk_eu = accuracy_score(y_test, y_pred_sk_eu)

sk_l1 = SkKNN(n_neighbors=5, metric="minkowski", p=1).fit(X_train_sc, y_train)  # Manhattan
y_pred_sk_l1 = sk_l1.predict(X_test_sc)
acc_sk_l1 = accuracy_score(y_test, y_pred_sk_l1)

print("\n=== sklearn.KNeighborsClassifier ===")
print(f"Accuracy (euclidean): {acc_sk_eu:.4f}")
print(f"Accuracy (manhattan): {acc_sk_l1:.4f}")



=== sklearn.KNeighborsClassifier ===
Accuracy (euclidean): 0.8440
Accuracy (manhattan): 0.8360


## Implementación de Regresión Lineal 📈🧮
La regresión lineal busca ajustar una recta (o hiperplano) que minimice los errores entre las predicciones y los valores reales.  
La solución se expresa de forma cerrada con la **fórmula vectorial de las betas**:

$$
\hat{\beta} = (X^\top X)^{-1} X^\top y
$$

Esta expresión surge de minimizar la suma de errores cuadráticos y nos permite calcular directamente los coeficientes del modelo.

In [ ]:

class LinearRegressionCustom(BaseEstimator, RegressorMixin):
    """
    Implementación básica de Regresión Lineal compatible con scikit-learn.
    Parámetros
    ----------
    fit_intercept : bool
        Si True, añade una columna de unos para el intercepto.
    """

    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept
        self.coef_ = None   # vector de coeficientes (sin intercepto si aplica)
        self.intercept_ = None  # valor escalar del intercepto si fit_intercept=True

    def fit(self, X, y):
        """
        Ajusta el modelo a los datos (X, y) resolviendo la ecuación normal:
        β = (X^T X)^(-1) X^T y
        - Si fit_intercept=True: añadir columna de unos a X.
        - Guardar coef_ y intercept_ según corresponda.
        - Devolver self.
        """
        # --- Validaciones y casting ---
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        if X.ndim != 2:
            raise ValueError("X debe ser 2D (n_samples, n_features).")
        if y.ndim != 1:
            raise ValueError("y debe ser 1D (n_samples,).")
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"Incompatibilidad: X tiene {X.shape[0]} filas y y tiene {y.shape[0]} elementos.")
        if np.isnan(X).any() or np.isnan(y).any():
            raise ValueError("X o y contienen NaNs. Limpia o imputa antes de ajustar.")

        # TODO: añadir columna de unos si fit_intercept
        X_aug = ####


        # TODO: resolver betas con np.linalg.pinv o np.linalg.solve
        beta_star = ####        
        
        
        # TODO: asignar self.coef_ y self.intercept_
        if self.fit_intercept:
            self.intercept_ = float(beta_star[0])
            self.coef_ = beta_star[1:].copy()
        else:
            self.intercept_ = 0.0
            self.coef_ = beta_star.copy()

        return self

    def predict(self, X):
        """
        Predice valores para nuevas muestras.
        - Añadir columna de unos si fit_intercept=True.
        - Calcular y = X @ beta.
        - Devuelve vector (n_samples,).
        """

        # --- Validaciones y casting ---
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("X debe ser 2D (n_samples, n_features).")
        if np.isnan(X).any():
            raise ValueError("X contiene NaNs. Limpia o imputa antes de predecir.")


        # TODO: validar que fit() haya sido llamado

        # TODO: añadir columna de unos si fit_intercept
        
        # TODO: calcular predicciones
        


    # ----------------------
    # Métodos auxiliares
    # ----------------------

    def _add_intercept(self, X):
        """
        Añade columna de unos a X si fit_intercept=True.
        """
        if not self.fit_intercept:
            return X
        n = X.shape[0]
        return np.c_[np.ones(n, dtype=float), X]

    def _check_is_fitted(self):
        """
        Verifica que fit() haya sido llamado (coef_ no None).
        """
        if self.coef_ is None or self.intercept_ is None:
            raise ValueError("Este LinearRegressionCustom no está ajustado. Llama primero a fit(X, y).")


In [41]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression as SkLinear

In [44]:
X, y,coef_ = make_regression(
    n_samples=500,
    n_features=5,
    noise=15.0,
    coef=True,
    random_state=42)


In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [46]:
lin_custom = LinearRegressionCustom(fit_intercept=True).fit(X_train, y_train)
y_pred_custom = lin_custom.predict(X_test)

r2_custom = r2_score(y_test, y_pred_custom)
mse_custom = mean_squared_error(y_test, y_pred_custom)

print("=== Regresión Lineal personalizada ===")
print(f"R2: {r2_custom:.4f}")
print(f"MSE: {mse_custom:.4f}")
print("Coeficientes:", lin_custom.coef_)
print("Intercepto:", lin_custom.intercept_)

=== Regresión Lineal personalizada ===
R2: 0.9800
MSE: 239.2313
Coeficientes: [28.68056303 81.95575413 31.06408509 68.81121129 10.89012328]
Intercepto: 0.06772465843754194


In [47]:
lin_sklearn = SkLinear(fit_intercept=True).fit(X_train, y_train)
y_pred_sk = lin_sklearn.predict(X_test)

r2_sk = r2_score(y_test, y_pred_sk)
mse_sk = mean_squared_error(y_test, y_pred_sk)

print("\n=== sklearn.LinearRegression ===")
print(f"R2: {r2_sk:.4f}")
print(f"MSE: {mse_sk:.4f}")
print("Coeficientes:", lin_sklearn.coef_)
print("Intercepto:", lin_sklearn.intercept_)


=== sklearn.LinearRegression ===
R2: 0.9800
MSE: 239.2313
Coeficientes: [28.68056303 81.95575413 31.06408509 68.81121129 10.89012328]
Intercepto: 0.06772465843753128


## Implementación del Ofuscador 🔒📊
Un **ofuscador de datos** aplica una transformación matricial a las características para proteger la información sin perder la posibilidad de recuperarla.  
La idea es multiplicar la matriz de datos $X$ por una **matriz reversible** $P$:

$$
X' = X \cdot P
$$

Dado que $P$ es invertible, podemos recuperar los datos originales con:

$$
X = X' \cdot P^{-1}
$$

Este mecanismo permite **ocultar los datos sensibles** en tránsito o en almacenamiento, conservando la posibilidad de revertir la transformación en entornos seguros.

In [53]:
rng = np.random.default_rng(0)
n = 3
A = rng.integers(low=0, high=n**2,size=(n, n))
A, np.linalg.det(A)

(array([[7, 5, 4],
        [2, 2, 0],
        [0, 0, 1]]),
 np.float64(4.000000000000001))

In [54]:
class FeatureObfuscator(BaseEstimator, TransformerMixin):
    """
    Ofuscador lineal reversible: X' = X @ P
    - Si P es invertible, se recupera X = X' @ P^{-1}.
    
    Parámetros
    ----------
    P : np.ndarray | None
        Matriz de ofuscación (dxd) proporcionada por el usuario (debe ser invertible).
        Si None, se generará en fit() .
    seed : int | None
        Semilla para reproducibilidad cuando se genere P.
    """

    def __init__(self,  P=None,  seed=None):
        self.P = P
        self.seed = seed
        # Atributos aprendidos en fit()
        self.P_ = None       # Matriz de ofuscación final (validada/generada)
        self.P_inv_ = None   # Inversa para inverse_transform

    # ----------------------
    # API sklearn
    # ----------------------

    def fit(self, X, y=None):
        """
        Valida/infere n_features y define P_ y P_inv_.
        - Si P es proporcionada: validar forma (dxd), invertibilidad y asignar.
        - Si no: generar P_  y calcular P_inv_.
        - Debe devolver self.
        """
        X = self._validate_2d(X)
        d = X.shape[1]
        rng = np.random.default_rng(self.seed)

        if self.P is not None:
            P = np.asarray(self.P, dtype=float)
            if P.shape != (d, d):
                raise ValueError(f"P debe tener forma {(d, d)}; recibida {P.shape}.")
            # Revisa si la matriz es invertible 
            if np.linalg.matrix_rank(P) < d:  # puede usarse  igual al determinante np.linalg.det(A)
                raise ValueError("La matriz P proporcionada no es invertible.")
            self.P_ = P.copy()
        else:
            self.P_ = self._generate_P(d, rng)

        # calcular inversa  de P 
            self.P_inv_ = np.linalg.inv(self.P_)

        return self

    def transform(self, X):
        """
        Aplica la ofuscación: X' = X @ P_
        - Validar que fit() fue llamado (P_ no None).
        - Validar dimensionalidad: X.shape[1] == P_.shape[0]
        - Devolver X ofuscada.
        """
        self._check_is_fitted()
        X = self._validate_2d(X)
        if X.shape[1] != self.P_.shape[0]:
            raise ValueError(f"Incompatibilidad: X tiene {X.shape[1]} columnas y P tiene {self.P_.shape[0]}.")
        return X @ self.P_

    def inverse_transform(self, X_obf):
        """
        Revierte la ofuscación: X = X_obf @ P_inv_
        - Validar que fit() fue llamado y que P_inv_ existe.
        - Devolver datos originales.
        """
        self._check_is_fitted()
        X_obf = self._validate_2d(X_obf)
        if X_obf.shape[1] != self.P_inv_.shape[0]:
            raise ValueError(f"Incompatibilidad: X_obf tiene {X_obf.shape[1]} columnas y P_inv tiene {self.P_inv_.shape[0]}.")
        return X_obf @ self.P_inv_

    # ----------------------
    # Utilidades internas
    # ----------------------

    def _generate_P(self, d, rng):
        """
        Genera una matriz P (dxd) invertible det!=0 .
        """        
        while True:
            A = rng.integers(low=0, high=d**2,size=(d, d))
            if np.linalg.det(A) > 1:
                break
        return A


    def _check_is_fitted(self):
        """
        Verifica que fit() haya sido llamado (P_ y P_inv_ no None).
        """
        if self.P_ is None or self.P_inv_ is None:
            raise ValueError("Este FeatureObfuscator no está ajustado. Llama primero a fit(X).")

    def _validate_2d(self, X):
        """
        Convierte a np.array float y valida que sea 2D (n_samples, n_features).
        """
        X = np.asarray(X, dtype=float)
        if X.ndim != 2:
            raise ValueError("Se espera una matriz 2D (n_samples, n_features).")
        if np.isnan(X).any():
            raise ValueError("X contiene NaNs; limpia o imputa antes de transformar.")
        return X


In [ ]:
# Instanciar y ajustar ofuscador en train
obf = FeatureObfuscator(seed=2025).fit(X_train)
X_train_obf = obf.transform(X_train)
X_test_obf  = obf.transform(X_test)

X_train.shape, X_train_obf.shape

((400, 5), (400, 5))

In [58]:
# Ajustar modelo en espacio ofuscado
lin_obf = SkLinear(fit_intercept=True).fit(X_train_obf, y_train)
yhat_obf = lin_obf.predict(X_test_obf)
r2_obf = r2_score(y_test, yhat_obf)
mse_obf = mean_squared_error(y_test, yhat_obf)

print("\n=== Modelo entrenado sobre datos ofuscados ===")
print(f"R2 : {r2_obf:.4f}")
print(f"MSE: {mse_obf:.4f}")


=== Modelo entrenado sobre datos ofuscados ===
R2 : 0.9800
MSE: 239.2313


In [59]:
# -------------------------------------------------------------------
# (4) Verificación de recuperación exacta
# -------------------------------------------------------------------
X_train_rec = obf.inverse_transform(X_train_obf)
X_test_rec  = obf.inverse_transform(X_test_obf)
rec_ok = np.allclose(X_train_rec, X_train) and np.allclose(X_test_rec, X_test)
print("\nRecuperación exacta de X via inverse_transform:", rec_ok)


Recuperación exacta de X via inverse_transform: True


In [60]:
# -------------------------------------------------------------------
# (5) Equivalencia de predicciones vía transformación de coeficientes
#     Si X' = X P y β' es entrenado en X', entonces β = P β' debe reproducir ŷ en X.
# -------------------------------------------------------------------
beta_prime = lin_obf.coef_           # coeficientes en espacio ofuscado
beta = obf.P_ @ beta_prime           # llevarlos al espacio original
intercept = lin_obf.intercept_

yhat_from_beta = X_test @ beta + intercept

print("\nEquivalencia de predicciones (Xβ vs X'β'):")
print("np.allclose(yhat_from_beta, yhat_obf) ->", np.allclose(yhat_from_beta, yhat_obf))


Equivalencia de predicciones (Xβ vs X'β'):
np.allclose(yhat_from_beta, yhat_obf) -> True
